# ADRIA Banking Platform - Détection de Fraudes Bancaires (Partie H)

Ce notebook présente l'analyse exploratoire, la préparation des données, la modélisation et l'évaluation du modèle de classification binaire pour la détection de transactions frauduleuses.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))
from fraud_pipeline import load_data, analyze_dataset, stratified_train_test_split, PreprocessingPipeline, LogisticRegressionModel, evaluate_classifier

## 1 & 2. Chargement des données et Analyse Exploratoire

In [2]:
data_path = os.path.abspath('../data/bank_transactions.csv')
rows = load_data(data_path)
prop_fraud, missing_counts = analyze_dataset(rows)

## 3. Séparation Entraînement / Test Stratifiée

In [3]:
train_rows, test_rows = stratified_train_test_split(rows, test_ratio=0.20, seed=42)
print(f'Train: {len(train_rows)} exemples, Test: {len(test_rows)} exemples')

## 4 & 5. Pipeline de Prétraitement (Exclusion des identifiants, Imputation, One-Hot Encoding, Normalisation)
> **Important** : Le prétraitement est ajusté uniquement sur le jeu d'entraînement.

In [4]:
pipeline = PreprocessingPipeline()
pipeline.fit(train_rows)
X_train, y_train = pipeline.transform(train_rows)
X_test, y_test = pipeline.transform(test_rows)
print('Features :', pipeline.feature_names)

## 6. Entraînement du Modèle (Régression Logistique avec pénalisation L2 et pondération)

In [5]:
model = LogisticRegressionModel(lr=0.1, epochs=400, l2_reg=0.01)
model.fit(X_train, y_train)

## 7. Évaluation sur l'Ensemble de Test

In [6]:
test_probs = model.predict_proba(X_test)
results = evaluate_classifier(y_test, test_probs, threshold=0.5)
for k, v in results.items():
    print(f'{k}: {v}')

## Réponses aux Questions d'Analyse Métier

### 1. Pourquoi l’accuracy seule est-elle insuffisante ?
Dans un contexte de détection de fraudes bancaires, les classes sont fortement déséquilibrées (ex. 96-99% de transactions légitimes pour 1-4% de fraudes). Un prédicteur naïf qui prédirait systématiquement « transaction légitime (0) » obtiendrait une accuracy apparente de 96-99%, tout en laissant passer **100% des fraudes**. L'accuracy masque donc l'incapacité totale du modèle à identifier les événements critiques de la classe minoritaire.

### 2. Quel compromis existe entre faux positifs et fraudes non détectées (faux négatifs) ?
- **Faux Positifs (FP)** : Une transaction légitime est bloquée à tort. Conséquence : friction pour le client, panier abandonné, mécontentement, surcharge du support client et coût opérationnel d'investigation manuelle.
- **Faux Négatifs (FN)** : Une véritable fraude n'est pas détectée. Conséquence : perte financière directe pour la banque et son client (chargeback), préjudice réputationnel et risques de non-conformité réglementaire.
- **Compromis** : Trouver le seuil optimal minimisant le coût global espéré (coût financier des FN vs coût d'expérience et d'investigation des FP).

### 3. Quel est l’effet possible d’une diminution du seuil de décision ?
En diminuant le seuil de décision (par exemple de 0.50 à 0.25) :
- On augmente le **rappel (recall)** : le modèle devient plus vigilant et capture davantage de fraudes réelles (baisse des FN).
- En contrepartie, la **précision diminue** et le nombre de **faux positifs (FP) augmente**, car des transactions légitimes suspectes seront signalées.

### 4. Pourquoi privilégier une validation temporelle sur des données bancaires réelles ?
Une séparation aléatoire classique (stratifiée ou k-fold) introduit un **data leakage temporel** (fuite d'informations futures vers le passé) et viole la causalité chronologique. En situation réelle :
- Les comportements des fraudeurs évoluent constamment (**concept drift** / dérive de concept).
- De nouveaux modes opératoires et marchands émergent.
- Une validation temporelle (ex. train sur les mois M à M+5, test sur le mois M+6) reproduit fidèlement les conditions réelles de production où le modèle doit généraliser sur des événements futurs inconnus.